# DeepWeeds Day 2 — Hồ Ngọc Mai (02509)
Run cells in order. Select models and all hyperparameters on **validation** only; run test once per final seed.

In [ ]:
!pip -q install timm openpyxl thop
!git clone --depth 1 https://github.com/ngmai2005/K4-Track4-Day2-Deeplearning-Advance.git /kaggle/working/lab || true
%cd /kaggle/working/lab/submissions/02509_ho_ngoc_mai
import sys, torch, timm
sys.path.insert(0, 'code'); sys.path.insert(0, '/kaggle/working/lab')
print(torch.__version__, timm.__version__, torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import hashlib, urllib.request, zipfile
DATA=Path('data'); LABELS=DATA/'labels'; DATA.mkdir(exist_ok=True); LABELS.mkdir(exist_ok=True)
archive=DATA/'images.zip'
if not archive.exists(): urllib.request.urlretrieve('https://zenodo.org/records/7939060/files/images.zip?download=1', archive)
h=hashlib.md5();
with archive.open('rb') as f:
    for c in iter(lambda:f.read(1<<20), b''): h.update(c)
assert h.hexdigest()=='b7b30f96d466fba86016aa5a26606e0f'
with zipfile.ZipFile(archive) as z: z.extractall(DATA)
for n in ('labels','train_subset0','val_subset0','test_subset0'):
    urllib.request.urlretrieve(f'https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels/{n}.csv', LABELS/f'{n}.csv')
IMAGES_DIR=DATA  # archive extracts its 17,509 JPEGs directly here
print('MD5',h.hexdigest(),'images',len(list(DATA.glob('*.jpg'))))

In [ ]:
from checks import main as run_checks
from dataset import load_split, check_split
run_checks()
tr,va,te=load_split(LABELS); split_stats=check_split(tr,va,te,IMAGES_DIR); split_stats

In [ ]:
from train import Config, run
# Run B01--B05 (same T00 recipe, seed 0). Record only validation metrics.
# After choosing a backbone from validation, run controlled T01--T08.
# Example: run(Config(exp_id='B01', backbone='resnet50', images_dir=str(IMAGES_DIR), labels_dir=str(LABELS)))
# Final only after configuration lock: F01 and T00, seeds 0/1/2, save_test_predictions=True.